# PySpark and Apache Spark: A Short Guide

This notebook is a **from-scratch to intermediate** guide to **Apache Spark** and **PySpark**—what they are, when to use them, and how to run basic data processing locally.

We use small, in-memory examples (similar in spirit to the FX-style data in other guides) so you can run everything in a single machine without a cluster.

**You will need:** Python 3.8+ and **Java (JDK 8 or 11+)** installed; PySpark runs on the JVM under the hood.

### Contents

- [1. What is Spark and PySpark?](#1-what-is-spark-and-pyspark)
- [2. Do I Need to Install Anything?](#2-do-i-need-to-install-anything)
- [3. Is There a Python Library?](#3-is-there-a-python-library)
- [4. SparkSession and Your First DataFrame](#4-sparksession-and-your-first-dataframe)
- [5. Lazy Evaluation and the Two Kinds of Operations](#5-lazy-evaluation-and-the-two-kinds-of-operations)
- [6. Transformations and Actions by Example](#6-transformations-and-actions-by-example)
- [7. Reading and Writing Data](#7-reading-and-writing-data)
- [8. When to Use Spark vs Pandas](#8-when-to-use-spark-vs-pandas)

---

## 1. What is Spark and PySpark?

**Apache Spark** is an open-source **distributed computing engine** for large-scale data processing. It was designed to:

- Process **huge datasets** that don’t fit on one machine by splitting work across many nodes.
- Offer **unified APIs** for batch processing, SQL, streaming, and machine learning.

**PySpark** is the **official Python API** for Spark. It lets you use Spark’s DataFrame and RDD APIs from Python. Under the hood, your Python code is sent to the JVM where Spark runs; data can stay distributed across the cluster.

**In short:** Spark = “distributed Pandas/SQL/streaming on a cluster”; PySpark = “use it from Python”.

---

## 2. Do I Need to Install Anything?

Yes, two things:

1. **Java** – Spark runs on the JVM. Install a JDK (e.g. OpenJDK 11 or 17) and set `JAVA_HOME` if needed. Check with `java -version` in a terminal.
2. **PySpark** – The Python package: `pip install pyspark`.

That’s enough to run Spark **locally** on your laptop (single process, or “local” mode). To run on a **cluster**, you’d typically use a managed service (Databricks, EMR, etc.) or set up Spark yourself on multiple machines; this guide sticks to local use.

---

## 3. Is There a Python Library?

Yes. The library is **PySpark**, and it is the official Python front-end for Apache Spark. After `pip install pyspark` you can:

- Import `pyspark.sql` (DataFrames, SQL, SparkSession).
- Import `pyspark.rdd` if you need the lower-level RDD API.

We will use the **DataFrame API** (`pyspark.sql`), which is the most common and Pandas-like.

In [ ]:
# Check that PySpark is available (and optionally Java)

try:
    import pyspark
    from pyspark.sql import SparkSession
    print("PySpark version:", pyspark.__version__)
except ImportError as e:
    print("PySpark not installed. Run: pip install pyspark")
    print(e)

# Java is required at runtime; SparkSession.builder.getOrCreate() will fail if Java is missing.

---

## 4. SparkSession and Your First DataFrame

In PySpark you start by creating a **SparkSession**. It is the entry point for DataFrame and SQL APIs. In local mode you typically do:

```python
spark = SparkSession.builder.appName("my-app").getOrCreate()
```

Then you can build DataFrames from Python lists/dicts, from files (CSV, Parquet, etc.), or from other sources. The next cell creates a small DataFrame from a list of rows and shows the schema and a few rows.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

spark = SparkSession.builder.appName("training").getOrCreate()

# Small dataset: symbol, price, size (like ticks)
rows = [
    Row(symbol="EURUSD", price=1.10, size=1_000_000),
    Row(symbol="GBPUSD", price=1.33, size=500_000),
    Row(symbol="EURUSD", price=1.11, size=2_000_000),
]

df = spark.createDataFrame(rows)
df.printSchema()
df.show()

---

## 5. Lazy Evaluation and the Two Kinds of Operations

Spark uses **lazy evaluation**: transformations (e.g. `filter`, `select`, `groupBy`) do not run immediately. They build a **logical plan**. Execution happens when you call an **action** (e.g. `show`, `collect`, `count`, `write`).

- **Transformation** – returns a new DataFrame (or Dataset); no data is read or computed yet.
- **Action** – triggers the plan to run and produces a result (or side effect like writing a file).

This allows Spark to optimize the full pipeline and, in a cluster, to distribute work.

---

## 6. Transformations and Actions by Example

We apply a few transformations (`filter`, `select`, `groupBy` + `agg`) and then trigger execution with actions (`show`, `collect`). This mirrors typical “filter → aggregate → show” workflows.

In [ ]:
# Transformations: filter and select
eur = df.filter(df.symbol == "EURUSD").select("symbol", "price", "size")

# Action: run the plan and show results
eur.show()

# Another transformation: groupBy + agg
from pyspark.sql import functions as F

summary = df.groupBy("symbol").agg(
    F.avg("price").alias("avg_price"),
    F.sum("size").alias("total_size"),
)

summary.show()

---

## 7. Reading and Writing Data

Spark can read/write many formats (CSV, Parquet, JSON, etc.). Here we write the small DataFrame to a CSV and read it back. Paths can be local or on HDFS/S3 in a cluster.

In [ ]:
import os

path = "training_spark_output"
if os.path.exists(path):
    import shutil
    shutil.rmtree(path)

df.write.mode("overwrite").csv(path, header=True)

read_back = spark.read.option("header", "true").csv(path)
read_back.show()

# Clean up (optional)
# if os.path.exists(path): shutil.rmtree(path)

---

## 8. When to Use Spark vs Pandas

| Use **Spark** when … | Use **Pandas** when … |
|------------------------|------------------------|
| Data is too large for one machine | Data fits in memory on one machine |
| You have a cluster or want to scale out | Single-node scripts or notebooks |
| You need SQL, streaming, or ML on big data | You need rich Python ecosystem (NumPy, SciPy, plotting) |

For small to medium data (e.g. &lt; 10 GB on a decent laptop), Pandas is usually simpler and faster. Spark shines when data or compute is distributed. You can also convert between them: `spark_df.toPandas()` and `spark.createDataFrame(pandas_df)`—but `toPandas()` brings data to the driver, so only for small results.

In [ ]:
# Optional: stop the session when done (e.g. in scripts)
# spark.stop()